# 3 - Download Census Data


**Variables I'm pulling (all from ACS 5-year, table family B):**

| Concept | Census table | How to compute |
|---|---|---|
| Median household income | `B19013_001E` | Direct |
| % bachelor's degree or higher | `B15003_022E..025E / B15003_001E` | Sum bach+master+prof+doc, divide by total 25+ |
| % below poverty | `B17001_002E / B17001_001E` | Direct |
| % non-Hispanic white | `B03002_003E / B03002_001E` | Direct |
| Median age | `B01002_001E` | Direct |
| Total population | `B01003_001E` | Direct (used for log-pop control + sanity check) |

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from current directory
KEY = os.getenv('CENSUS_API_KEY')
assert KEY, 'CENSUS_API_KEY not found. Did you copy .env.example to .env and fill it in?'
print('Key loaded:', KEY[:6] + '...' + KEY[-4:])

AssertionError: CENSUS_API_KEY not found. Did you copy .env.example to .env and fill it in?

In [ ]:
VARS = [
    'B19013_001E',                                              # median HH income
    'B15003_001E','B15003_022E','B15003_023E','B15003_024E','B15003_025E',  # education
    'B17001_001E','B17001_002E',                                # poverty
    'B03002_001E','B03002_003E',                                # race (NH white)
    'B01002_001E',                                              # median age
    'B01003_001E',                                              # total population
]

url = 'https://api.census.gov/data/2022/acs/acs5'
params = {
    'get': 'NAME,' + ','.join(VARS),
    'for': 'county:*',
    'in':  'state:*',
    'key': KEY,
}
print('Calling Census ACS 2022 5-year API for all US counties...')
r = requests.get(url, params=params, timeout=60)
r.raise_for_status()
data = r.json()
print('Got', len(data) - 1, 'rows (counties + DC + PR if included)')

In [ ]:
# Build dataframe (first row of API response is headers)
raw = pd.DataFrame(data[1:], columns=data[0])

# Numeric conversion (API returns everything as strings)
for v in VARS:
    raw[v] = pd.to_numeric(raw[v], errors='coerce')

# Build 5-digit FIPS
raw['fips'] = raw['state'].astype(str).str.zfill(2) + raw['county'].astype(str).str.zfill(3)
raw.head()

### Compute derived percentages

In [ ]:
census = pd.DataFrame()
census['fips'] = raw['fips']
census['name'] = raw['NAME']
census['median_hh_income']     = raw['B19013_001E']
census['median_age']           = raw['B01002_001E']
census['population']           = raw['B01003_001E']
census['log_population']       = np.log10(raw['B01003_001E'].clip(lower=1))

# % bachelor's or higher (of pop 25+)
census['pct_bachelors_plus'] = (
    (raw['B15003_022E'] + raw['B15003_023E'] + raw['B15003_024E'] + raw['B15003_025E'])
    / raw['B15003_001E'] * 100
)
# % below poverty
census['pct_poverty'] = raw['B17001_002E'] / raw['B17001_001E'] * 100
# % non-Hispanic white
census['pct_nhwhite'] = raw['B03002_003E'] / raw['B03002_001E'] * 100

# Drop Puerto Rico (state FIPS 72) - CHR data is 50 states + DC only
census = census[~census['fips'].str.startswith('72')].reset_index(drop=True)
print('Census rows after dropping PR:', len(census))
census.head()

In [ ]:
# Sanity-check distributions
summary_cols = ['median_hh_income','pct_bachelors_plus','pct_poverty','pct_nhwhite','median_age','population']
census[summary_cols].describe().round(1)

Expected ranges:
- `median_hh_income`: ~$25k to ~$165k, median ~$60k
- `pct_bachelors_plus`: 5% to 80%, median ~22%
- `pct_poverty`: 2% to 45%, median ~13%
- `pct_nhwhite`: 0% to ~98%, median ~75% (skewed heavily white because most US counties are demographically white)
- `median_age`: 22 to 67, median ~41


In [ ]:
os.makedirs('data', exist_ok=True)
census.to_csv('data/census_df.csv', index=False)
print('Wrote data/census_df.csv -', len(census), 'counties')

**Next:** notebook 4 - merge outcomes with Census demographics.